In [1]:
from pathlib import Path
from pypdf import PdfReader
import sys
sys.path.append(str(Path.cwd().parent))
from src.chunker import clean_text, chunker

In [2]:
pdf = PdfReader(Path("../data/samplerag.pdf"))
raw_pages=[]
for i,page in enumerate(pdf.pages):
    text = page.extract_text()
    if text:
        raw_pages.append(text)
    else:
        print(f"{i+1} has no text")
raw_text="\n".join(raw_pages)
cleaned_te = clean_text(raw_text)
chunks= chunker(cleaned_te,500,100)
print(chunks[0])

Think of RAG as: “Let the model look things up before it answers.” A beginner-friendly one-line definition: Retrieval-Augmented Generation (RAG) is a system design where an LLM first retrieves relevant information from a knowledge base, then uses that information to generate a grounded answer. The key intuition is this: • A normal LLM answers from what it learned during training • A RAG system answers from both o the model’s general knowledge o fresh, specific, private, or domain-specific docume


In [3]:
chunk_metadata = []
for i,chunk in enumerate(chunks):
    chunk_metadata.append({"chunk_id":i, "text":chunk,
                           "source":"samplerag.pdf","length":len(chunk)})
chunk_metadata[0]

{'chunk_id': 0,
 'text': 'Think of RAG as: “Let the model look things up before it answers.” A beginner-friendly one-line definition: Retrieval-Augmented Generation (RAG) is a system design where an LLM first retrieves relevant information from a knowledge base, then uses that information to generate a grounded answer. The key intuition is this: • A normal LLM answers from what it learned during training • A RAG system answers from both o the model’s general knowledge o fresh, specific, private, or domain-specific docume',
 'source': 'samplerag.pdf',
 'length': 500}

In [4]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

d:\Downloads\Desktop\AI\rag-from-scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 789.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
chunk_embeddings = embedding_model.encode(chunks,show_progress_bar=True,convert_to_numpy=True)

print(f"Embedding matrix shape (chunks,dimensions):{chunk_embeddings.shape} ")

Batches: 100%|██████████| 2/2 [00:02<00:00,  1.07s/it]

Embedding matrix shape (chunks,dimensions):(49, 384) 


In [6]:
import faiss
import numpy as np

In [7]:
###faiss usually wants float-32

chunk_embeddings = np.array(chunk_embeddings).astype("float32")
faiss.normalize_L2(chunk_embeddings)

embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings)
print(f"Number of vectors in the index:{index.ntotal}")

Number of vectors in the index:49


In [8]:
query="What is rag?"

query_embedding = embedding_model.encode([query],convert_to_numpy=True).astype("float32")
faiss.normalize_L2(query_embedding)

print(query_embedding.shape)

(1, 384)


In [9]:
top_k=5

scores, indices = index.search(query_embedding,top_k)
print("Scores",scores)
print("Indices:", indices)

Scores [[0.7205193  0.54161495 0.49180198 0.488393   0.4839164 ]]
Indices: [[ 0 41 22 23  5]]


In [10]:
for rank, chunk_idx in enumerate(indices[0],start=1):
    print(f"\n---Rank {rank} | Chunk ID: {chunk_idx}---")
    print(chunk_metadata[chunk_idx]["text"])


---Rank 1 | Chunk ID: 0---
Think of RAG as: “Let the model look things up before it answers.” A beginner-friendly one-line definition: Retrieval-Augmented Generation (RAG) is a system design where an LLM first retrieves relevant information from a knowledge base, then uses that information to generate a grounded answer. The key intuition is this: • A normal LLM answers from what it learned during training • A RAG system answers from both o the model’s general knowledge o fresh, specific, private, or domain-specific docume

---Rank 2 | Chunk ID: 41---
tters because RAG is often implemented as an orchestration pipeline around the LLM. A strong whiteboard-style explanation You could explain it like this in an interview: There are two spaces in RAG: • the knowledge store where document chunks live as embeddings • the generation step where the LLM uses retrieved chunks to answer The bridge between them is the retriever. That’s a very nice conceptual framing. A good answer to “why embedding

In [11]:
questions=["what is ingestion?","what is vector database?","what is reranking?","what are limitations of rag"]

questions_embeddings = embedding_model.encode(questions,convert_to_numpy=True).astype("float32")
faiss.normalize_L2(questions_embeddings)
print(questions_embeddings.shape)

top_k=3
scores,indices = index.search(questions_embeddings,top_k)
print(indices.shape)

for i in range(indices.shape[0]):
    print(f"\n{questions[i]}")
    for rank, chunk_idx in enumerate(indices[i],start=1):
        print(f"\n--------rank: {rank} || chunk-id:{chunk_idx}-----")
        print(f"\n{chunk_metadata[chunk_idx]["text"]}")

(4, 384)
(4, 3)

what is ingestion?

--------rank: 1 || chunk-id:40-----

nstructions • retrieved context • user question LLM Reads the prompt and writes the answer. Output Final user-facing response. One subtle but important point The LLM usually does not search the vector DB directly by itself. Usually the application orchestrates it: • app receives user query • app calls embedding model • app queries retriever/vector DB • app constructs prompt • app calls LLM This matters because RAG is often implemented as an orchestration pipeline around the LLM. A strong whitebo

--------rank: 2 || chunk-id:8-----

 split them into chunks 3. convert each chunk into an embedding 4. store embeddings + chunk text + metadata in a vector database So there are really two phases in RAG: A. Offline / ingestion phase Prepare the knowledge base B. Online / query phase Handle the user’s question This distinction is very useful in interviews. Phase A: Offline / ingestion phase 1. Documents Suppose you have: 

# Generation

In [12]:
def retrieve_chunks(query,top_k=3):
    qembed= embedding_model.encode([query],convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(qembed)
    scores,indices = index.search(qembed,top_k)
    results=[]
    for score, chunk_idx in zip(scores[0],indices[0]):
        results.append({
            "chunk_id":int(chunk_idx),
            "score":float(score),
            "text":chunk_metadata[chunk_idx]["text"],
            "source":chunk_metadata[chunk_idx]["source"]
        })
    return results

In [13]:
res = retrieve_chunks("What is RAG?",3)

for r in res:
    print("\n")
    print(f"chunk_ id:{r["chunk_id"]}")
    print(f"Score:{r["score"]}")
    print(f"\n{r["text"]}")



chunk_ id:0
Score:0.7205193042755127

Think of RAG as: “Let the model look things up before it answers.” A beginner-friendly one-line definition: Retrieval-Augmented Generation (RAG) is a system design where an LLM first retrieves relevant information from a knowledge base, then uses that information to generate a grounded answer. The key intuition is this: • A normal LLM answers from what it learned during training • A RAG system answers from both o the model’s general knowledge o fresh, specific, private, or domain-specific docume


chunk_ id:41
Score:0.5416149497032166

tters because RAG is often implemented as an orchestration pipeline around the LLM. A strong whiteboard-style explanation You could explain it like this in an interview: There are two spaces in RAG: • the knowledge store where document chunks live as embeddings • the generation step where the LLM uses retrieved chunks to answer The bridge between them is the retriever. That’s a very nice conceptual framing. A good 

In [24]:
#Augumentation: Taking the retrieved chunks and inserting them into prompt as external context.

def build_prompt(query, retrieve_chunks):
    context_blocks=[]
    for i, chunk in enumerate(retrieve_chunks,start=1):
        context_blocks.append(
            f"[Context {i} | Chunk ID: {chunk["chunk_id"]} | Score: {chunk["score"]}]\n{chunk["text"]}"
        )

    context_text='\n'.join(context_blocks)
    prompt = f"""
You are a helpful assistant answering questions using only the provided context.

Instructions:
- Use the context below to answer the user's question.
- If the answer is not clearly supported by the context, say: "I don't know based on the provided context."
- Keep the answer clear and concise.
- Do not invent facts outside the context.

Retrieved Context:{context_text}
User Question:{query}
Answer:""".strip()

    return prompt
    




In [25]:
prompt = build_prompt("What is RAG?",res)

print(prompt[:3000])

You are a helpful assistant answering questions using only the provided context.

Instructions:
- Use the context below to answer the user's question.
- If the answer is not clearly supported by the context, say: "I don't know based on the provided context."
- Keep the answer clear and concise.
- Do not invent facts outside the context.

Retrieved Context:[Context 1 | Chunk ID: 0 | Score: 0.7205193042755127]
Think of RAG as: “Let the model look things up before it answers.” A beginner-friendly one-line definition: Retrieval-Augmented Generation (RAG) is a system design where an LLM first retrieves relevant information from a knowledge base, then uses that information to generate a grounded answer. The key intuition is this: • A normal LLM answers from what it learned during training • A RAG system answers from both o the model’s general knowledge o fresh, specific, private, or domain-specific docume
[Context 2 | Chunk ID: 41 | Score: 0.5416149497032166]
tters because RAG is often imple